In [3]:
import pandas as pd
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)
print (df.head(5))
print(df.describe())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
  

## Cuál es la diferencia matemática entre el percentil 75% (el tercer cuartil) y el valor Máximo (max)? Como ingenieros de software, ¿qué les dice esta enorme dispersión estadística sobre la homogeneidad de los precios que se cobraron en el barco?
_Nos dice que el 25% de las personas les cobraron más de 31 dolares. Como ingeniero de software, eso nos dice que hay outliers en nuestros datos, lo cual dice que a algunos les cobraron una cantidad desproporcionada._

## ¿Qué anomalía de base de datos acaban de descubrir?
_Datos nulos o no validos._
##  El algoritmo que construirán al final del semestre requiere que todos los pasajeros tengan un valor numérico en la columna Age para poder compilar. Sabiendo lo que acaban de descubrir en la Pregunta B, propongan al menos dos estrategias a nivel de base de datos o de programación para resolver este problema antes de alimentar a la IA.

1. El primer approach es rellenar la edad calculando la media de edad según la clase del pasajero (Pclass) para ser más precisos. 
2. Si la cantidad de filas con edades nulas representa un porcentaje estadísticamente insignificante del total de su base de datos_




In [3]:
lista_pasajeros = df.to_dict(orient='records')

suma_mujeres = 0.0
cuenta_mujeres = 0

suma_hombres = 0.0
cuenta_hombres = 0

for pasajero in lista_pasajeros:
    if pasajero['Sex'] == 'female':
        suma_mujeres += pasajero['Fare']
        cuenta_mujeres += 1
    elif pasajero['Sex'] == 'male':
        suma_hombres += pasajero['Fare']
        cuenta_hombres += 1

promedio_mujeres = suma_mujeres / cuenta_mujeres
promedio_hombres = suma_hombres / cuenta_hombres

print(f"Tarifa promedio de las mujeres: {promedio_mujeres:.2f}")
print(f"Tarifa promedio de los hombres: {promedio_hombres:.2f}")


Tarifa promedio de las mujeres: 44.48
Tarifa promedio de los hombres: 25.52


In [6]:
df.groupby('Sex')["Fare"].mean()

Sex
female    44.479818
male      25.523893
Name: Fare, dtype: float64

In [5]:
fare_series = df['Fare']

q25 = fare_series.quantile(0.25)
q75 = fare_series.quantile(0.75)

fare_IQR = q75 - q25
minimum = q25 - 1.5 * fare_IQR
maximum = q75 + 1.5 * fare_IQR

print("IQR:", fare_IQR)
print("Minimum:", minimum)
print("Maximum:", maximum)


IQR: 23.0896
Minimum: -26.724
Maximum: 65.6344


In [7]:
##Pregunta A (Sumarización Categórica):
print("Pregunta A (Sumarización Categórica):")
print(df['Survived'].value_counts(normalize=True) * 100)
## Pregunta B (Agrupación y Agregación):
print("Pregunta B (Agrupación y Agregación):")
print(df.groupby('Sex')['Survived'].value_counts(normalize=True)*100)
## Pregunta C:
print("Pregunta C:")
outliers = df[df['Fare']> maximum]
print(outliers.groupby('Pclass').size())
## Pregunta D:
print("Pregunta D:")
print("Media: ",df['Fare'].mean())
print("Mediana: ",df['Fare'].median())






muertos = df[df['Survived'] == 0]
vivos = df[df['Survived'] == 1]

muestra_muertos = muertos.sample(n=62, random_state=1)

muestra_vivos = vivos.sample(n=38, random_state=1)

muestra_final = pd.concat([muestra_muertos, muestra_vivos])

print(df.groupby('Survived')['Age'].mean())

print("\nPregunta E")
sobrevivientes = df[df["Survived"] == 1]
no_sobrevivientes = df[df["Survived"] == 0]
proporcion_sobrevivientes = len(sobrevivientes) / len(df)
n_sobrevivientes = round(150 * proporcion_sobrevivientes)
n_no_sobrevivientes = 150 - n_sobrevivientes
muestra_sobrevivientes = sobrevivientes.sample(n=n_sobrevivientes, random_state=1)
muestra_no_sobrevivientes = no_sobrevivientes.sample(n=n_no_sobrevivientes, random_state=1)
muestra = pd.concat([muestra_sobrevivientes, muestra_no_sobrevivientes])
print(f"Tamano de la muestra: {len(muestra)}")
print(muestra["Survived"].value_counts(normalize=True))
print(df["Survived"].value_counts(normalize=True))

print("\nPregunta F")
edad_por_supervivencia = df.groupby("Survived")["Age"].mean()
print(edad_por_supervivencia)
print(f"NaN en Age: {df['Age'].isna().sum()}")
print(df[df["Age"].isna()].groupby(["Pclass", "Survived"]).size())

print("\nPregunta G")
print(outliers[["Name", "Pclass", "Fare"]])

print("\nPregunta H")
df["Titulo"] = df["Name"].str.extract(r",\s*([^\.]*)\.")
print(df["Titulo"].value_counts())
print(df.groupby("Titulo")["Age"].median())

print("\nPregunta I")
varianza_survived = df["Survived"].var()
print(f"Varianza: {varianza_survived:.4f}")

print("\nPregunta J")
grupos = df.groupby(["Pclass", "Sex", "Embarked"])["PassengerId"].count()
print(grupos)
print(f"Grupos con 1 o 2 pasajeros: {(grupos <= 2).sum()} de {len(grupos)}")


Pregunta A (Sumarización Categórica):
Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64
Pregunta B (Agrupación y Agregación):
Sex     Survived
female  1           74.203822
        0           25.796178
male    0           81.109185
        1           18.890815
Name: proportion, dtype: float64
Pregunta C:
Pclass
1    104
2      5
3      7
dtype: int64
Pregunta D:
Media:  32.204207968574636
Mediana:  14.4542
Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

Pregunta E
Tamano de la muestra: 150
Survived
0    0.613333
1    0.386667
Name: proportion, dtype: float64
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Pregunta F
Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64
NaN en Age: 177
Pclass  Survived
1       0            16
        1            14
2       0             7
        1             4
3       0           102
        1            34
dtype: int64

Pregunta G
                                            

Pregunta A: La tasa de supervivencia global fue de 38.38%. O sea que de todos los pasajeros del barco, solo poco más de 1 de cada 3 sobrevivió.

Pregunta B: Sí se cumple el código "mujeres y niños primero", al menos con las mujeres: el 74.20% de las mujeres sobrevivió, mientras que solo el 18.89% de los hombres sobrevivió. Es una diferencia enorme, casi 4 veces más probabilidad de sobrevivir siendo mujer.

Pregunta C: Q1 = 7.91, Q3 = 31.00, entonces el IQR = 23.09. El límite superior aceptable queda en 65.63. Filtrando el DataFrame con ese límite, salen 116 pasajeros que pagaron una tarifa por encima de eso, y la gran mayoría (104 de 116) eran de primera clase (Pclass 1). Tiene sentido porque los boletos más caros eran justamente los de lujo.

Pregunta D: La media de Fare es 32.20 y la mediana es 14.45. Que la media sea tanto más alta que la mediana significa que la distribución está sesgada a la derecha (hay unos pocos boletos carísimos que jalan el promedio hacia arriba, aunque la mayoría de la gente pagó bastante menos). Si usara KNN sin escalar esta variable, el modelo le daría muchísimo más peso a Fare que a otras variables (como Age).

Pregunta E: El código separa a los sobrevivientes y no sobrevivientes, calcula qué proporción representa cada grupo en el dataset completo, y saca esa misma proporción pero sobre 150 pasajeros (los resultados quedaron en 61.3% / 38.7%, casi idéntico al 61.6% / 38.4% original). Esto evita el sesgo de muestreo: si sacara una muestra completamente al azar, podría por mala suerte agarrar muchos más sobrevivientes que no sobrevivientes (o viceversa), y el modelo aprendería con una proporción de clases que no refleja la realidad.

Pregunta F: Si la mayoría de los NaN de Age pertenecen a pasajeros de 3ra clase que murieron, y Pandas simplemente los ignora al calcular la media, entonces esa media de edad para el grupo "Survived=0" está calculada con menos datos de 3ra clase de los que debería. Esto introduce un sesgo de selección, el promedio de edad de los que murieron termina reflejando más a los pasajeros de 1ra y 2da clase (donde sí se registró la edad) que a los de 3ra. El modelo podría aprender un patrón de edad que no representa bien a la población real, sobre todo a los pasajeros más pobres.

Pregunta G: No, no deberíamos simplemente eliminarlos con .drop(). El Titanic era un barco de lujo con pasajeros muy adinerados (viendo los nombres, muchos son "Mrs." con apellidos de familias ricas de la época), así que boletos de 500+ libras no son errores de captura, son casos reales y válidos (señal, no ruido). Eliminarlos le quitaría al modelo información real sobre cómo la clase social/económica influía en la supervivencia. Lo correcto seria conservarlos, y si el algoritmo lo requiere (como KNN), transformar o escalar la variable (por ejemplo con una transformación logarítmica) en lugar de borrar los datos.

Pregunta H: Sería superior porque el título captura información que la edad global no captura: un "Master" es un niño, mientras que "Mr" es un adulto. Si imputáramos con la mediana global (28 años aproximadamente), le estaríamos poniendo esa edad tanto a los niños con título "Master" como a los adultos "Mr", lo cual es claramente incorrecto. Al agrupar por título, cada grupo faltante se imputa con un valor mucho más cercano a la realidad (los "Master" con mediana de 3.5 años, los "Mr" con 30 años, etc.), reduciendo el error que se introduce al inventar esos datos faltantes.

Pregunta I: La varianza de Survived salió en 0.2368, cercana a lo esperado. Si la varianza fuera exactamente 0.0, significaría que todos los pasajeros tienen el mismo valor en Survived (o todos sobrevivieron o todos murieron, no hay ninguna variación). Si intentáramos entrenar un modelo de clasificación con una variable de respuesta así, sería imposible: el modelo no tendría nada que aprender ni que distinguir, porque no existe ningún patrón que separe clases distintas — literalmente solo hay una clase.

Pregunta J: Salieron 4 grupos con 1 o 2 pasajeros (de 18 grupos en total). Si dejamos que el modelo aprenda reglas basadas en esos grupos tan chiquitos, va a ocurrir sobreajuste (overfitting). Con solo 1 o 2 casos, el modelo memorizaría esos casos específicos como si fueran una regla general, cuando en realidad es una muestra demasiado pequeña para sacar ninguna conclusión estadísticamente confiable. Eso hace que el modelo funcione muy bien con esos datos de entrenamiento pero generalice mal con datos nuevos.